# A pretrained backbone: feature extraction, then fine-tuning

The same 2,000 images, and 97% — because the model has already seen 1.4 million others. This is the most valuable technique in the chapter and the one with the strictest procedure.

**Runs on:** GPU recommended — about 20 minutes on CPU · needs the Kaggle cats-vs-dogs archive &nbsp;·&nbsp; **Slides:** [Chapter 8 — Image Classification](../../../course-web-slides/ch08/index.html) &nbsp;·&nbsp; **Section:** 04 — Using a pretrained model

---

## Loading a backbone without its head

In [ ]:
import keras
from keras import layers

conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False,          # drop the 1000-class classifier
    input_shape=(180, 180, 3),
)
conv_base.summary()
print(f"\n{conv_base.count_params():,} parameters, all pretrained")

`include_top=False` drops the ImageNet classifier and keeps the **convolutional base** — the part that learned edges, textures, and object parts. Those transfer; the 1000-class head does not.

## Fast feature extraction, without augmentation

Run every image through the frozen base once, cache the features, and train a small classifier on those. **Very fast, and it rules out augmentation** — the features are computed once, so there is nothing to randomise.

In [ ]:
import numpy as np
import pathlib
from keras.utils import image_dataset_from_directory

new_base_dir = pathlib.Path("cats_vs_dogs_small")
train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=(180, 180), batch_size=32)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=(180, 180), batch_size=32)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=(180, 180), batch_size=32)

def get_features_and_labels(dataset):
    all_features, all_labels = [], []
    for images, labels in dataset:
        preprocessed = keras.applications.vgg16.preprocess_input(images)
        features = conv_base.predict(preprocessed, verbose=0)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)
print(train_features.shape)

Expected output:

```
(2000, 5, 5, 512)
```

In [ ]:
inputs = keras.Input(shape=(5, 5, 512))
x = layers.Flatten()(inputs)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])

callbacks = [keras.callbacks.ModelCheckpoint(
    "feature_extraction.keras", save_best_only=True, monitor="val_loss")]
h1 = model.fit(train_features, train_labels, epochs=20,
               validation_data=(val_features, val_labels),
               callbacks=callbacks, verbose=2)
print("test:", keras.models.load_model("feature_extraction.keras")
      .evaluate(test_features, test_labels, verbose=0)[1])

Expected output:

```
test: ~0.97
```

**Ninety-seven percent, in twenty seconds of training.** Against 70% from scratch and 83% with augmentation. Everything the base knows was learned before this notebook started.

## Feature extraction with augmentation

Slower, because the base runs on every batch — but augmentation works again, and the ceiling is higher.

In [ ]:
conv_base.trainable = False
print("trainable weights after freezing:", len(conv_base.trainable_weights))

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
])

inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = keras.applications.vgg16.preprocess_input(x)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])

> ⚠️ **`conv_base.trainable = False` **before** compiling.** Setting it afterwards has no effect on an already-compiled model, and the base will train — destroying the weights you came for, with no error.

## Fine-tuning: the procedure, in order

Unfreezing part of the base can add a few more points. The order is not negotiable.

1. Add your head to a **frozen** base.
2. Train the head to convergence.
3. Unfreeze the **top few** layers of the base.
4. Retrain both, at a **much lower** learning rate.

Skip step 2 and the large, random gradients from an untrained head propagate into the base and wreck it on the first batch.

In [ ]:
conv_base.trainable = True
for layer in conv_base.layers[:-4]:
    layer.trainable = False

for layer in conv_base.layers:
    print(f"  {layer.name:24s} trainable={layer.trainable}")

In [ ]:
model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              metrics=["accuracy"])

callbacks = [keras.callbacks.ModelCheckpoint(
    "fine_tuning.keras", save_best_only=True, monitor="val_loss")]
h2 = model.fit(train_dataset, epochs=30,
               validation_data=validation_dataset,
               callbacks=callbacks, verbose=2)

best = keras.models.load_model("fine_tuning.keras")
print(f"test accuracy: {best.evaluate(test_dataset, verbose=0)[1]:.3f}")

`learning_rate=1e-5` — a hundred times smaller than the default. The same discipline reappears in chapter 15 for RoBERTa and chapter 16 for Gemma. **Large updates destroy representations that cost a great deal to learn.**

## Why only the top layers

In [ ]:
import matplotlib.pyplot as plt

names = [l.name for l in conv_base.layers if "conv" in l.name]
print("Earlier layers encode generic features -- edges, colours, textures.")
print("Later layers encode specific ones -- 'dog ear', 'car wheel'.")
print()
for i, n in enumerate(names):
    kind = "generic (keep frozen)" if i < len(names) - 3 else "specific (worth tuning)"
    print(f"  {n:16s} {kind}")

Two reasons to leave the early layers alone: they encode features that transfer to *any* image problem, and every unfrozen layer is more parameters to fit from 2,000 samples. **More trainable parameters on a small dataset is more overfitting**, which is the thing you were trying to fix.

## The whole chapter, in one table

In [ ]:
print(f"{'approach':38s} {'test acc':>9s}")
print(f"{'-'*48}")
for name, acc in [("from scratch", 0.70),
                  ("+ augmentation + dropout", 0.83),
                  ("VGG16 features, cached", 0.97),
                  ("VGG16 + augmentation", 0.975),
                  ("VGG16 fine-tuned (top 4 layers)", 0.98)]:
    print(f"{name:38s} {acc:>9.3f}")
print("\n(your numbers will vary by a point or two)")

The largest single jump is **from scratch to pretrained**, and it is not close. Chapter 15 makes the same point about text and chapter 16 about generation. If there is one habit to take from this chapter: ==start from a pretrained backbone, always, and justify not doing so.==

---

## What to take away

- `include_top=False` keeps the convolutional base and drops the task-specific head.
- Cached features are fastest but rule out augmentation; running the base per batch costs time and restores it.
- Freeze **before** compiling, or the base trains and is destroyed.
- Fine-tune only the top layers, only after the head has converged, and only at a much lower learning rate.